<a href="https://colab.research.google.com/github/ben854719/Autonomous-Security-Orchestration-Layer/blob/main/Autonomous_Security_Orchestration_layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
%%writefile dashboard.js

// -----------------------------
// DOM Elements
// -----------------------------
const telemetryPanel = document.getElementById("telemetry");
const systemStatusPanel = document.getElementById("systemStatus");
const packetPanel = document.getElementById("packet");
const rulePathPanel = document.getElementById("rulePath");
const loadButton = document.getElementById("loadPacket");
const ctx = document.getElementById('telemetryChart').getContext('2d');

// -----------------------------
// Initialize Chart.js
// -----------------------------
let telemetryChart = new Chart(ctx, {
    type: 'line',
    data: {
        labels: [],
        datasets: [{
            label: 'Threat Velocity',
            borderColor: '#61afef',
            data: [],
            fill: false,
            tension: 0.4
        }, {
            label: 'Anomaly Density',
            borderColor: '#e06c75',
            data: [],
            fill: false,
            tension: 0.4
        }]
    },
    options: {
        responsive: true,
        maintainAspectRatio: false,
        scales: {
            y: { beginAtZero: true, grid: { color: '#3e4451' } },
            x: { grid: { display: false } }
        },
        plugins: {
            legend: { labels: { color: '#abb2bf' } }
        }
    }
});

// -----------------------------
// Render Telemetry & Update Chart
// -----------------------------
function renderTelemetry(data) {
  if (telemetryPanel) {
    telemetryPanel.innerHTML = `
      <h3>Live Telemetry</h3>
      <ul>
        <li><b>Threat Velocity:</b> ${data.threatVelocity}</li>
        <li><b>Anomaly Density:</b> ${data.anomalyDensity}</li>
        <li><b>Behavioral Drift:</b> ${data.behavioralDrift}</li>
        <li><b>Immune Response Time:</b> ${data.immuneResponseTime}</li>
        <li><b>System Stability:</b> ${data.systemStability}</li>
      </ul>
    `;
  }

  // Update Chart Data
  const timeLabel = new Date().toLocaleTimeString();
  telemetryChart.data.labels.push(timeLabel);
  telemetryChart.data.datasets[0].data.push(data.threatVelocity);
  telemetryChart.data.datasets[1].data.push(data.anomalyDensity);

  // Keep only last 10 points
  if (telemetryChart.data.labels.length > 10) {
    telemetryChart.data.labels.shift();
    telemetryChart.data.datasets[0].data.shift();
    telemetryChart.data.datasets[1].data.shift();
  }
  telemetryChart.update();
}

// -----------------------------
// Render System Status
// -----------------------------
function renderSystemStatus() {
  if (systemStatusPanel) {
    systemStatusPanel.innerHTML = `
      <h3>System Status</h3>
      <p><b>ACIS Mode:</b> Autonomous Defense</p>
      <p><b>Policy Profile:</b> acis.v1.production</p>
      <p><b>Last Update:</b> ${new Date().toLocaleTimeString()}</p>
    `;
  }
}

// -----------------------------
// RS256 Verification (Stub)
// -----------------------------
function verifyPacket(packet) {
  return true;
}

// -----------------------------
// Render Packet
// -----------------------------
function renderPacket(packet) {
  if (packetPanel && rulePathPanel) {
    const verified = verifyPacket(packet);
    packetPanel.innerHTML = `
      <h3>Latest Diagnostic Packet</h3>
      <pre>${JSON.stringify(packet, null, 2)}</pre>
      <p class="${verified ? "verified" : "failed"}">
        ${verified ? "RS256 Signature Verified ✔" : "Verification Failed ✖"}
      </p>
    `;
    rulePathPanel.innerHTML = `
      <h3>Rule Path Trace</h3>
      <p>${packet.rulePath}</p>
    `;
  }
}

// -----------------------------
// Simulation Loop
// -----------------------------
if (loadButton) {
  loadButton.addEventListener("click", () => {
    const packet = {
      immuneResponseScore: 92,
      diagnosticId: "diag-" + Date.now(),
      rulePath: "acis.v1.immune.response.trace",
      timestamp: new Date().toISOString()
    };
    renderPacket(packet);
  });
}

// Simulate incoming data every 3 seconds
setInterval(() => {
  renderTelemetry({
    threatVelocity: (Math.random() * 0.5 + 0.2).toFixed(2),
    anomalyDensity: (Math.random() * 0.2 + 0.1).toFixed(2),
    behavioralDrift: "Stable",
    immuneResponseTime: "1.4s",
    systemStability: "Nominal"
  });
  renderSystemStatus();
}, 3000);

// Initial Render
renderTelemetry({
  threatVelocity: 0.42,
  anomalyDensity: 0.18,
  behavioralDrift: "Stable",
  immuneResponseTime: "1.4s",
  systemStability: "Nominal"
});

Overwriting dashboard.js


In [5]:
import os

# Ensure the dashboard variable is defined in this scope
final_dash_html = """
<div id='acis-final-container' style='width: 100%; height: 600px; background: #050505; border-radius: 20px; overflow: hidden; position: relative; border: 1px solid #1a1a1a; font-family: monospace;'>
    <div style='position: absolute; top: 20px; left: 20px; z-index: 10; pointer-events: none;'>
        <h2 style='color: #00f2ff; margin: 0; text-shadow: 0 0 10px #00f2ff;'>ACIS CORE VISUALIZER</h2>
        <div id='stats-overlay' style='color: #fff; background: rgba(0,0,0,0.85); padding: 15px; border-radius: 10px; border-left: 3px solid #00f2ff; margin-top: 15px; min-width: 250px;'>
            <div style='display: flex; justify-content: space-between;'><span>THREAT VELOCITY:</span> <span id='v-3d' style='color: #ff0055; font-weight: bold;'>0.00</span></div>
            <div style='display: flex; justify-content: space-between;'><span>ANOMALY DENSITY:</span> <span id='d-3d' style='color: #00f2ff;'>0.00</span></div>
        </div>
    </div>
    <div id='three-canvas-container' style='width: 100%; height: 100%;'></div>
</div>

<script src='https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js'></script>
<script>
(function() {
    let scene, camera, renderer, mesh, points;
    function init3D() {
        const container = document.getElementById('three-canvas-container');
        if (!container) return;
        scene = new THREE.Scene();
        camera = new THREE.PerspectiveCamera(75, container.clientWidth / container.clientHeight, 0.1, 1000);
        renderer = new THREE.WebGLRenderer({ antialias: true, alpha: true });
        renderer.setSize(container.clientWidth, container.clientHeight);
        container.appendChild(renderer.domElement);
        const geometry = new THREE.IcosahedronGeometry(2, 4);
        const material = new THREE.MeshPhongMaterial({ color: 0x00f2ff, wireframe: true, transparent: true, opacity: 0.3, emissive: 0x00f2ff });
        mesh = new THREE.Mesh(geometry, material);
        scene.add(mesh);
        camera.position.z = 6;
        function animate() { requestAnimationFrame(animate); mesh.rotation.x += 0.003; mesh.rotation.y += 0.003; renderer.render(scene, camera); }
        animate();
    }
    if (typeof THREE !== 'undefined') init3D();
    else { const check = setInterval(() => { if (typeof THREE !== 'undefined') { clearInterval(check); init3D(); } }, 100); }
})();
</script>
"""

# Generate the standalone HTML file
standalone_html = f"""<!DOCTYPE html>
<html>
<head>
    <title>ACIS 3D Operational Dashboard</title>
    <style>
        body, html {{ margin: 0; padding: 0; width: 100%; height: 100%; background: #050505; overflow: hidden; }}
    </style>
</head>
<body>
    {final_dash_html}
</body>
</html>
"""

with open('acis_dashboard_standalone.html', 'w') as f:
    f.write(standalone_html)

print("Standalone HTML file 'acis_dashboard_standalone.html' has been created.")

Standalone HTML file 'acis_dashboard_standalone.html' has been created.


In [6]:
from IPython.display import HTML, display

# Advanced 3D ACIS Operational Dashboard - Refined Visual Logic
final_dash_html = """
<div id='acis-final-container' style='width: 100%; height: 600px; background: #050505; border-radius: 20px; overflow: hidden; position: relative; border: 1px solid #1a1a1a; font-family: monospace;'>
    <div style='position: absolute; top: 20px; left: 20px; z-index: 10; pointer-events: none;'>
        <h2 style='color: #00f2ff; margin: 0; text-shadow: 0 0 10px #00f2ff;'>ACIS CORE VISUALIZER</h2>
        <div id='stats-overlay' style='color: #fff; background: rgba(0,0,0,0.85); padding: 15px; border-radius: 10px; border-left: 3px solid #00f2ff; margin-top: 15px; min-width: 250px;'>
            <div style='display: flex; justify-content: space-between;'><span>THREAT VELOCITY:</span> <span id='v-3d' style='color: #ff0055; font-weight: bold;'>0.00</span></div>
            <div style='display: flex; justify-content: space-between;'><span>ANOMALY DENSITY:</span> <span id='d-3d' style='color: #00f2ff;'>0.00</span></div>
            <div style='margin-top: 10px; font-size: 10px; color: #00f2ff; border-top: 1px solid #333; padding-top: 5px;'>STATE: PARADOXICAL EFFICIENCY</div>
        </div>
    </div>
    <div id='three-canvas-container' style='width: 100%; height: 100%;'></div>
</div>

<script src='https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js'></script>
<script>
(function() {
    let scene, camera, renderer, mesh, points, frameId;

    function init3D() {
        const container = document.getElementById('three-canvas-container');
        if (!container) return;

        scene = new THREE.Scene();
        camera = new THREE.PerspectiveCamera(75, container.clientWidth / container.clientHeight, 0.1, 1000);
        renderer = new THREE.WebGLRenderer({ antialias: true, alpha: true });
        renderer.setSize(container.clientWidth, container.clientHeight);
        renderer.setPixelRatio(window.devicePixelRatio);
        container.appendChild(renderer.domElement);

        const geometry = new THREE.IcosahedronGeometry(2, 4);
        const material = new THREE.MeshPhongMaterial({
            color: 0x00f2ff,
            wireframe: true,
            transparent: true,
            opacity: 0.3,
            emissive: 0x00f2ff,
            emissiveIntensity: 0.5
        });
        mesh = new THREE.Mesh(geometry, material);
        scene.add(mesh);

        const pGeom = new THREE.BufferGeometry();
        const pCount = 800;
        const positions = new Float32Array(pCount * 3);
        for(let i=0; i<pCount*3; i++) positions[i] = (Math.random() - 0.5) * 12;
        pGeom.setAttribute('position', new THREE.BufferAttribute(positions, 3));
        const pMat = new THREE.PointsMaterial({ color: 0x00f2ff, size: 0.03 });
        points = new THREE.Points(pGeom, pMat);
        scene.add(points);

        const light = new THREE.PointLight(0xffffff, 1, 100);
        light.position.set(5, 5, 5);
        scene.add(light);

        camera.position.z = 6;

        function animate() {
            frameId = requestAnimationFrame(animate);
            mesh.rotation.x += 0.003;
            mesh.rotation.y += 0.003;
            points.rotation.y -= 0.001;

            const time = Date.now() * 0.001;
            const pulse = 1 + Math.sin(time * 3) * 0.1;
            mesh.scale.set(pulse, pulse, pulse);

            renderer.render(scene, camera);
        }
        animate();

        setInterval(() => {
            const v = (0.65 + Math.random() * 0.25).toFixed(2);
            const d = (0.08 + Math.random() * 0.07).toFixed(2);

            const vElem = document.getElementById('v-3d');
            const dElem = document.getElementById('d-3d');
            if(vElem) vElem.innerText = v;
            if(dElem) dElem.innerText = d;

            if (v > 0.75) {
                mesh.material.color.setHex(0xff0055);
                mesh.material.emissive.setHex(0xff0055);
                points.material.color.setHex(0xffaa00);
            } else {
                mesh.material.color.setHex(0x00f2ff);
                mesh.material.emissive.setHex(0x00f2ff);
                points.material.color.setHex(0x00f2ff);
            }
        }, 1500);
    }

    if (typeof THREE !== 'undefined') init3D();
    else {
        const check = setInterval(() => {
            if (typeof THREE !== 'undefined') {
                clearInterval(check);
                init3D();
            }
        }, 100);
    }
})();
</script>
"""
display(HTML(final_dash_html))

In [7]:
!pip install --upgrade langchain-google-genai google-generativeai langgraph==1.1.1 langsmith

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.8/507.8 kB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 67.5 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.7.23
    Uninstalling langsmith-0.7.23:
      Successfully uninstalled langsmith-0.7.23
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.23
    Uninstalling langchain-core-1.2.23:
      Successfully uninstalled langchain-core-1.2.23
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.4
    Uninstalling langgraph-1.1.4:
      Successfully uninstalled langgraph-1.1.4


In [8]:
import os
os.environ["LANGSMITH_API_KEY"] = "LangSmith"

In [9]:
!pip install "mcp[cli]"
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("GeminiTools")

@mcp.tool()
def search(query: str) -> list:
    # Your search logic here
    return ["Result 1", "Result 2"]

In [10]:
!cd mcp-server-demo

/bin/bash: line 1: cd: mcp-server-demo: No such file or directory


In [11]:
!cd mcp-server-demo && uv add langchain-google-genai langgraph langsmith

/bin/bash: line 1: cd: mcp-server-demo: No such file or directory


In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, END
from IPython.display import display, HTML
from typing import TypedDict, List
import os
from mcp.server.fastmcp import FastMCP
from google.colab import userdata
from langsmith import traceable
import random
import json

# ACIS Agent State Definition

class AgentState(TypedDict):
    input: str
    logs: str
    symptom: str
    diagnostic_report: str
    context: dict
    network_ok: bool
    cache_ok: bool
    token_valid: bool
    version_ok: bool
    kpi_metrics: dict
    trace_id: str
    threat_model: dict
    resilience_model: dict
    immune_response: dict

# ACIS Prompt Template (Aligned With Your Architecture).

prompt = ChatPromptTemplate.from_template("""
You are the Agentic Reasoning Layer of the Autonomous Cyber Immune System (ACIS).

Core Principles:
- Self‑Evolving Threat Response: Detect anomalies, synthesize digital antibodies, and adapt to behavioral drift.
- Simulation‑Driven Resilience Modeling: Evaluate systemic risk, bottlenecks, and resilience scores.
- Transparent Diagnostic Reasoning: Produce traceable, audit‑ready explanations.
- Human‑Aligned Decision Support: Provide adaptive insights without overriding human judgment.

Signal/Symptom: {symptom}
System Logs: {logs}

Performance Metrics (Current vs Baseline):
{kpi_data}

Your tasks:
1. Analyze cyber‑immune health for anomalies, drift, or degradation.
2. Evaluate KPI improvements and operational efficiency.
3. Integrate threat‑response modeling and resilience simulation outputs.
4. Produce a diagnostic report with root causes, impacts, and transparent reasoning.
""")

# Gemini Model Binding.

api_key = userdata.get("Ben856")
os.environ["GOOGLE_API_KEY"] = api_key

llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    api_key=api_key
)

try:
    tools = [search]
    llm_with_tools = llm.bind_tools(tools)
except NameError:
    llm_with_tools = llm

#  Entry Node — ACIS Context Synthesis.

@traceable(name="entry_node")
def entry_node(state: AgentState) -> AgentState:

    # KPI metrics derived from ACIS telemetry
    kpi_summary = {
        "Threat Velocity": {"current": 0.42, "baseline": 0.30},
        "Anomaly Density": {"current": 0.18, "baseline": 0.25},
        "Immune Response Time (sec)": {"current": 1.4, "baseline": 2.1},
        "System Stability": {"current": 0.92, "baseline": 0.80}
    }

    # ACIS threat‑response engine output
    threat_model = {
        "detected": True,
        "type": "Behavioral Drift",
        "confidence": 0.87,
        "antibody_synthesized": True,
        "deployment_time_ms": 420
    }

    # ACIS resilience simulation output
    resilience_model = {
        "resilience_score": 0.91,
        "bottlenecks": ["Network Latency", "Resource Allocation"],
        "risk_projection": "Low",
        "timeline_projection_min": 12
    }

    # ACIS immune‑response packet
    immune_response = {
        "digital_antibody": "ACIS-ABX-2026",
        "response_vector": "behavioral-correction",
        "effectiveness": 0.93
    }

    return {
        **state,
        "symptom": state["input"],
        "kpi_metrics": kpi_summary,
        "threat_model": threat_model,
        "resilience_model": resilience_model,
        "immune_response": immune_response,
        "trace_id": f"trace-{random.randint(1000, 9999)}"
    }

# LLM Node — ACIS Agentic Reasoning Layer.

@traceable(name="llm_node")
def llm_node(state: AgentState) -> AgentState:

    kpi_str = json.dumps(state["kpi_metrics"], indent=2)

    messages = prompt.format_messages(
        symptom=state["symptom"],
        logs=state.get("logs", "Normal operational telemetry"),
        kpi_data=kpi_str
    )

    response = llm_with_tools.invoke(messages)

    return {
        **state,
        "diagnostic_report": response.content
    }

# Display Node — ACIS Intelligence Output.

@traceable(name="display_node")
def display_node(state: AgentState) -> AgentState:

    print("\n=== ACIS Autonomous Cyber‑Immune Intelligence Report ===")
    print(f"Trace ID: {state['trace_id']}")
    print(f"Context Source: {state['context'].get('source', 'ACIS Dashboard')}")

    print("\n--- Threat Model ---")
    print(json.dumps(state["threat_model"], indent=2))

    print("\n--- Resilience Simulation ---")
    print(json.dumps(state["resilience_model"], indent=2))

    print("\n--- Immune Response Packet ---")
    print(json.dumps(state["immune_response"], indent=2))

    print("\n--- Agentic Diagnostic Report (Gemini) ---")
    print(state["diagnostic_report"])

    return state

# Workflow Graph.
workflow = StateGraph(AgentState)
workflow.add_node("entry", entry_node)
workflow.add_node("llm", llm_node)
workflow.add_node("display", display_node)

workflow.set_entry_point("entry")
workflow.add_edge("entry", "llm")
workflow.add_edge("llm", "display")
workflow.add_edge("display", END)

agent_executor = workflow.compile()

# Execute ACIS Agent.

agent_executor.invoke({
    "input": "Analyze the latest ACIS Dashboard metrics showing elevated threat velocity.",
    "logs": "Detected behavioral drift and latency bottlenecks.",
    "context": {"source": "ACIS Operational Dashboard"}
})


=== ACIS Autonomous Cyber‑Immune Intelligence Report ===
Trace ID: trace-7381
Context Source: ACIS Operational Dashboard

--- Threat Model ---
{
  "detected": true,
  "type": "Behavioral Drift",
  "confidence": 0.87,
  "antibody_synthesized": true,
  "deployment_time_ms": 420
}

--- Resilience Simulation ---
{
  "resilience_score": 0.91,
  "bottlenecks": [
    "Network Latency",
    "Resource Allocation"
  ],
  "risk_projection": "Low",
  "timeline_projection_min": 12
}

--- Immune Response Packet ---
{
  "digital_antibody": "ACIS-ABX-2026",
  "response_vector": "behavioral-correction",
  "effectiveness": 0.93
}

--- Agentic Diagnostic Report (Gemini) ---
[{'type': 'text', 'text': '### **ACIS Diagnostic Report: Agentic Reasoning Layer**\n**Date:** 2024-05-22\n**Status:** Active Mitigation / Adaptive Synthesis\n**Subject:** Analysis of Elevated Threat Velocity and Systemic Behavioral Drift\n\n---\n\n#### **1. Cyber-Immune Health Analysis**\nDespite the high-pressure environment, the AC

{'input': 'Analyze the latest ACIS Dashboard metrics showing elevated threat velocity.',
 'logs': 'Detected behavioral drift and latency bottlenecks.',
 'symptom': 'Analyze the latest ACIS Dashboard metrics showing elevated threat velocity.',
 'diagnostic_report': [{'type': 'text',
   'text': '### **ACIS Diagnostic Report: Agentic Reasoning Layer**\n**Date:** 2024-05-22\n**Status:** Active Mitigation / Adaptive Synthesis\n**Subject:** Analysis of Elevated Threat Velocity and Systemic Behavioral Drift\n\n---\n\n#### **1. Cyber-Immune Health Analysis**\nDespite the high-pressure environment, the ACIS ecosystem displays a **High-Resilience/High-Stress** state.\n\n*   **Behavioral Drift Detected:** The system logs indicate a deviation from the established behavioral baseline. While "System Stability" is high (0.92), the drift suggests the environment is evolving to accommodate new operational parameters—likely a result of the ACIS adapting its internal logic to counter the increased Threat